# 04 · Preparación y modelado — ENCC 2017

Este cuaderno continúa el análisis descriptivo de P6. Evalúa si se puede anticipar la asistencia al cine durante el último año (`p63`) a partir de características demográficas y hábitos digitales. Compara una línea de base con una regresión logística, con ponderación de encuesta y un flujo de preparación que evita fuga de datos. Parte de `datos/derivados/encc_2017_limpio.parquet`, generado por `02_limpieza_encc.ipynb`.

## Pregunta y objetivo

**Pregunta:** ¿qué tan bien se puede anticipar si una persona asistió al cine durante el último año (`p63`) usando su edad, sexo, región, nivel socioeconómico, estudios y hábitos digitales?

Este cuaderno continúa el análisis descriptivo de P6. La clasificación es una herramienta para resumir asociaciones predictivas; **no permite concluir causalidad** ni reemplaza el análisis ponderado ya realizado. La partición se hace antes de imputar, escalar o codificar. Cada transformación aprende sus parámetros exclusivamente del entrenamiento.

## Decisiones de preparación

- **Respuesta:** `p63` (`SI` / `NO`). Se excluyen las preguntas que dependen de esa respuesta (`p63_1`, `p63_2`) para no filtrar indirectamente el resultado.
- **Predictores fijados por la pregunta:** `edad`, `sexo`, `region`, `NSEcat1`, `p106` y los minutos diarios de internet, redes y videojuegos. `id` y `pondera_dem` nunca son predictores.
- `NSEcat1` y `p106` tienen orden, pero sus distancias no son necesariamente iguales. Se codifican como indicadores, sin inventar una distancia numérica.
- Los minutos tienen muchos ceros y cola derecha. Se aplica `log1p`, que conserva los ceros y comprime los valores grandes; después se escala de forma robusta.
- No se aplica PCA ni selección automática: el conjunto reducido de predictores responde a la pregunta sustantiva y conserva interpretabilidad.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler

SEMILLA = 2026
ARCHIVO = 'encc_2017_limpio.parquet'
candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p / 'proyecto_equipo' for p in candidatas
             if (p / 'proyecto_equipo/datos/derivados' / ARCHIVO).is_file()), None)
if RAIZ is None:
    RAIZ = next(p for p in candidatas if (p / 'datos/derivados' / ARCHIVO).is_file())
enc = pd.read_parquet(RAIZ / 'datos/derivados' / ARCHIVO)
assert len(enc) == 2800
print(f'Base limpia: {enc.shape[0]:,} filas × {enc.shape[1]} columnas')

Base limpia: 2,800 filas × 443 columnas


## 1. Respuesta y partición

Se conserva el diseño de 400 casos por región. La partición mantiene juntas, en proporciones similares, las combinaciones de región y respuesta. Reservamos el 25 % para una única evaluación final. No se usa esa partición para elegir variables, parámetros ni umbrales.

In [2]:
OBJETIVO = 'p63'
PESO = 'pondera_dem'
NUM_TIEMPOS = ['minutos_internet_total', 'minutos_redes_total',
               'minutos_videojuegos_total']
CATEGORICAS = ['sexo', 'region', 'NSEcat1', 'p106']
PREDICTORES = ['edad', *NUM_TIEMPOS, *CATEGORICAS]

datos = enc.loc[enc[OBJETIVO].isin(['SI', 'NO'])].copy()
y = datos[OBJETIVO].astype(str)
X = datos[PREDICTORES].copy()
pesos = pd.to_numeric(datos[PESO], errors='raise').astype(float)

print('Proporción sin ponderar:')
display(y.value_counts(normalize=True).rename('proporción').to_frame())
print('Proporción ponderada:')
display((pesos.groupby(y).sum() / pesos.sum()).rename('proporción ponderada').to_frame())

estrato = datos['region'].astype(str) + ' · ' + y
X_ent, X_pru, y_ent, y_pru, w_ent, w_pru = train_test_split(
    X, y, pesos, test_size=0.25, random_state=SEMILLA, stratify=estrato
)
print(f'Entrenamiento: {len(X_ent)} filas | Prueba: {len(X_pru)} filas')

Proporción sin ponderar:


,proporción
p63,
NO,0.604286
SI,0.395714


Proporción ponderada:


,proporción ponderada
p63,
NO,0.646021
SI,0.353979


Entrenamiento: 2100 filas | Prueba: 700 filas


## 2. Flujo de preparación

La imputación, transformación, escalado y codificación están dentro de `Pipeline`. Al ajustar el modelo, cada paso aprende del entrenamiento y luego transforma la prueba con esos mismos parámetros. `handle_unknown='ignore'` permite categorías nuevas sin romper el flujo. La regularización de la regresión logística evita que la codificación indicadora completa vuelva inestable el ajuste.

In [3]:
pipeline_edad = Pipeline([
    ('imputar', SimpleImputer(strategy='median', add_indicator=True)),
    ('escalar', RobustScaler())
])
pipeline_tiempos = Pipeline([
    ('imputar', SimpleImputer(strategy='median', add_indicator=True)),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('escalar', RobustScaler())
])
pipeline_categoricas = Pipeline([
    ('imputar', SimpleImputer(strategy='most_frequent')),
    ('indicadoras', OneHotEncoder(handle_unknown='ignore', drop='first'))
])
preparacion = ColumnTransformer([
    ('edad', pipeline_edad, ['edad']),
    ('tiempos', pipeline_tiempos, NUM_TIEMPOS),
    ('categorias', pipeline_categoricas, CATEGORICAS)
])
modelo = Pipeline([
    ('preparacion', preparacion),
    ('clasificador', LogisticRegression(max_iter=2000, C=1.0))
])

## 3. Línea de base y modelo

La línea de base predice siempre la clase más frecuente. Se compara con una regresión logística regularizada. `pondera_dem` se usa como peso de ajuste, no como predictor; normalizamos los pesos de entrenamiento para que su media sea uno y mantener una escala estable de regularización.

In [4]:
w_ent_ajuste = w_ent / w_ent.mean()
base = DummyClassifier(strategy='prior')
base.fit(X_ent, y_ent, sample_weight=w_ent_ajuste)
modelo.fit(X_ent, y_ent, clasificador__sample_weight=w_ent_ajuste);

## 4. Evaluación final

Las métricas se calculan con los pesos de encuesta para que la estimación refleje mejor la composición poblacional. El AUC evalúa la ordenación de probabilidades; la exactitud balanceada da el mismo peso a ambas clases. La matriz de confusión muestra proporciones ponderadas dentro de cada clase real. Con una sola partición, los resultados tienen variación muestral; no se ajusta el umbral mirando la prueba.

In [5]:
def medir(nombre, estimador, X_eval, y_eval, w_eval):
    clases = list(estimador.classes_)
    prob_si = estimador.predict_proba(X_eval)[:, clases.index('SI')]
    pred = estimador.predict(X_eval)
    return {
        'modelo': nombre,
        'exactitud ponderada': accuracy_score(y_eval, pred, sample_weight=w_eval),
        'exactitud balanceada ponderada': balanced_accuracy_score(
            y_eval, pred, sample_weight=w_eval),
        'precision SI ponderada': precision_score(
            y_eval, pred, pos_label='SI', sample_weight=w_eval, zero_division=0),
        'recall SI ponderado': recall_score(
            y_eval, pred, pos_label='SI', sample_weight=w_eval, zero_division=0),
        'F1 SI ponderado': f1_score(
            y_eval, pred, pos_label='SI', sample_weight=w_eval, zero_division=0),
        'AUC ROC ponderado': roc_auc_score(
            y_eval, prob_si, sample_weight=w_eval),
        'precisión promedio ponderada': average_precision_score(
            (y_eval == 'SI').astype(int), prob_si, sample_weight=w_eval)
    }

metricas = pd.DataFrame([
    medir('Línea de base', base, X_pru, y_pru, w_pru),
    medir('Regresión logística', modelo, X_pru, y_pru, w_pru)
]).set_index('modelo')
display(metricas.round(3).T)

pred_prueba = modelo.predict(X_pru)
matriz = confusion_matrix(y_pru, pred_prueba, labels=['NO', 'SI'],
                          sample_weight=w_pru, normalize='true')
display(pd.DataFrame(matriz, index=['Real NO', 'Real SI'],
                     columns=['Predicho NO', 'Predicho SI']).round(3))

modelo,Línea de base,Regresión logística
exactitud ponderada,0.637,0.716
exactitud balanceada ponderada,0.500,0.667
precision SI ponderada,0.000,0.643
recall SI ponderado,0.000,0.489
F1 SI ponderado,0.000,0.555
AUC ROC ponderado,0.500,0.773
precisión promedio ponderada,0.363,0.608


,Predicho NO,Predicho SI
Real NO,0.845,0.155
Real SI,0.511,0.489


## 5. Lectura del modelo

Los coeficientes describen cambios en la log-odds de asistir al cine, condicionados a los demás predictores. En cada variable categórica, el coeficiente se compara con la categoría de referencia que eligió el codificador. No es una medida causal ni sustituye las tablas ponderadas del cuaderno 03.

In [6]:
nombres = modelo.named_steps['preparacion'].get_feature_names_out()
coeficientes = pd.Series(
    modelo.named_steps['clasificador'].coef_[0], index=nombres, name='coeficiente logit'
)
coeficientes = coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index)
display(coeficientes.head(20).to_frame().round(3))

,coeficiente logit
categorias__region_NEA,-1.125
categorias__p106_SIN ESTUDIOS,-1.025
edad__edad,-1.020
categorias__NSEcat1_D2,-0.986
tiempos__minutos_redes_total,0.900
categorias__region_NOA,-0.886
categorias__NSEcat1_D1,-0.713
categorias__region_CUYO,-0.693
categorias__region_PBA,-0.679
categorias__p106_PRIMARIA INCOMPLETA,-0.655


## Resultado e interpretación

En esta partición reservada, la regresión logística obtuvo **exactitud ponderada 0,716**, frente a **0,637** de la línea de base; el **AUC ROC ponderado fue 0,773**, frente a 0,500. La precisión promedio ponderada fue 0,608, frente a 0,363. El modelo aporta señal predictiva sobre la línea de base.

Con el umbral predeterminado de 0,5, la precisión ponderada para `SI` fue 0,643 y el recall 0,489: detecta cerca de la mitad de las asistencias reales ponderadas y deja escapar la otra mitad. La matriz normalizada muestra recall 0,846 para `NO` y 0,489 para `SI`. El desempeño depende del umbral; no se lo ajustó usando la prueba. Estos resultados son predictivos, no causales, y provienen de una sola partición. Para estimar variabilidad, el siguiente paso sería validación cruzada repetida dentro del entrenamiento.

## Decisiones para la bitácora

| # | Decisión | Motivo | Resultado / seguimiento |
|---|---|---|---|
| 12 | Definir `p63` como respuesta y excluir `p63_1` y `p63_2` | Son respuestas posteriores o condicionadas a la asistencia; podrían revelar el objetivo | Se eligió por continuidad con P6; validar el alcance con el equipo |
| 13 | Usar predictores demográficos y digitales fijados antes de partir | Mantiene una pregunta acotada; excluye `id` y `pondera_dem` de las entradas | Mantener fija para evitar selección mirando la prueba |
| 14 | Aprender imputación, escalado y codificación dentro del flujo | Evita que la prueba influya en el preprocesamiento | Hecho: la transformación está dentro del flujo y no ve la prueba |
| 15 | Comprimir minutos con `log1p` y escalar con mediana/IQR | Hay muchos ceros y colas largas; no se eliminan casos extremos legítimos | No se hizo comparación de variantes; si se compara, hacerlo solo en entrenamiento |

**Conclusión:** en esta partición el modelo supera la línea de base en métricas ponderadas, con recall limitado para la clase `SI`. La generalización debe confirmarse con validación cruzada repetida en entrenamiento. La encuesta tiene ponderadores, pero este cuaderno no incorpora estratos ni conglomerados al cálculo de incertidumbre; no se reportan intervalos de confianza. La encuesta tiene ponderadores, pero este cuaderno no incorpora estratos ni conglomerados al cálculo de incertidumbre; no se reportan intervalos de confianza.